# RAG-Based PDF Question Answering System

## Overview

This project demonstrates a Retrieval-Augmented Generation (RAG) pipeline for intelligent question answering over PDF documents using Large Language Models (LLMs).

The system retrieves relevant document chunks using semantic vector search and generates grounded answers using a Groq-hosted Llama 3.3 model.

The application also includes:
- Source citations with page references
- Prompt-based guardrails to reduce hallucinations
- Semantic retrieval using FAISS vector database
- Context-aware answer generation

---

## Objective

To build a production-aware AI document assistant capable of:
- Understanding unstructured PDF documents
- Retrieving contextually relevant information
- Generating grounded answers using an LLM
- Preventing hallucinated responses using guardrails

---

## How It Works

1. Upload PDF document
2. Split text into smaller chunks
3. Convert chunks into vector embeddings
4. Store embeddings in FAISS vector database
5. Retrieve relevant chunks using semantic similarity
6. Pass retrieved context to the LLM
7. Generate grounded answers with source citations

---

## Tech Stack

- Python
- LangChain
- FAISS
- HuggingFace Embeddings
- Groq API
- Llama 3.3 70B Versatile
- PyPDFLoader

---

## Key Features

- Retrieval-Augmented Generation (RAG)
- Semantic document search
- LLM-powered question answering
- Source citation with page numbers
- Hallucination reduction using prompt guardrails
- Context-grounded response generation

---

## Example Query

"What is the Transformer architecture?"

---

## Example Output

Returns:
- AI-generated answer
- Retrieved contextual information
- Source page citations
- Refusal response for unrelated questions

In [1]:
!pip install langchain openai faiss-cpu tiktoken pypdf

In [2]:
from google.colab import files
uploaded = files.upload()

Saving NIPS-2017-attention-is-all-you-need-Paper.pdf to NIPS-2017-attention-is-all-you-need-Paper (1).pdf


In [3]:
!pip install langchain-community

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("NIPS-2017-attention-is-all-you-need-Paper.pdf")
documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 11


In [5]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Chunks created:", len(docs))

Chunks created: 11


In [6]:
!pip install sentence-transformers

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = FAISS.from_documents(docs, embeddings)

print("Vector database created (FREE)")

/tmp/ipykernel_21634/2976347028.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created (FREE)


In [9]:
!pip install langchain-groq

In [8]:
retriever = db.as_retriever()

query = "What is the Transformer architecture?"

results = retriever.invoke(query)

for r in results:
    print(r.page_content)
    print("------")

Figure 1: The Transformer - model architecture.
wise fully connected feed-forward network. We employ a residual connection [10] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined 

In [10]:
from langchain_groq import ChatGroq

In [11]:
llm = ChatGroq( groq_api_key = "********************************", model_name = "llama-3.3-70b-versatile")


In [12]:
response = llm.invoke("what is Machine Learning")
print(response.content)

**Machine Learning (ML)** is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time, without being explicitly programmed.

**Key Characteristics of Machine Learning:**

1. **Learning from Data**: ML algorithms learn from data, rather than being explicitly programmed.
2. **Improving Performance**: ML models improve their performance over time, as they receive more data and learn from their mistakes.
3. **Pattern Recognition**: ML algorithms recognize patterns in data, which enables them to make predictions or decisions.
4. **Autonomy**: ML models can operate autonomously, without human intervention.

**Types of Machine Learning:**

1. **Supervised Learning**: The model is trained on labeled data, where the correct output is already known.
2. **Unsupervised Learning**: The model is trained on unlabeled data, and it must find patterns or structure in 

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the question ONLY using the provided context.

If the answer is not present in the context, say:
"I could not find the answer in the document."

Do not make up information.
Do not use outside knowledge.

Context:
{context}

Question:
{question}
"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is the Transformer architecture?"))

The Transformer architecture is a sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. It consists of an encoder and a decoder, each composed of a stack of identical layers. The encoder contains self-attention layers, while the decoder contains self-attention layers and encoder-decoder attention layers. The model also uses positional encodings, embeddings, and softmax functions. The architecture is described in more detail in Figure 1 and throughout the text.


In [24]:
query = "What is the Transformer architecture?"

results = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in results])

response = llm.invoke(
    f"""
Answer the question based on the context below.

Context:
{context}

Question:
{query}
"""
)

print(response.content)

print("\nSources:")
pages = sorted(set(doc.metadata['page'] + 1 for doc in results))

for page in pages:
    print(f"Page: {page}")

The Transformer architecture is a sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. It consists of an encoder and a decoder, each composed of a stack of identical layers. The encoder contains self-attention layers, while the decoder contains self-attention layers and encoder-decoder attention layers. The model also uses positional encoding to inject information about the relative or absolute position of the tokens in the sequence. The Transformer architecture is designed to handle sequence-to-sequence tasks, such as machine translation, and has been shown to achieve state-of-the-art results on several benchmarks. 

The key components of the Transformer architecture include:

1. Encoder: The encoder is composed of a stack of identical layers, each consisting of two sub-layers: self-attention and position-wise feed-forward networks.
2. Decoder: The decoder is also c

In [27]:
print(chain.invoke("Who won FIFA World Cup 2022?"))

I could not find the answer in the document.
